<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/03_soil_terrain_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Soil, terrain and salinity features

**Purpose:** Add soil properties, terrain features and soil salinity indicators to every grid cell from notebook 01, for use as model inputs.

**Inputs:**
- `grid.parquet` from notebook 01 (~2 km cells, nine Sahel countries)
- Google Earth Engine: SoilGrids 250 m v2.0 (`projects/soilgrids-isric/*_mean`), SRTM 30 m (`USGS/SRTMGL1_003`), Global Soil Salinity Maps 1986-2016 (`projects/sat-io/open-datasets/global_soil_salinity`, Ivushkin et al. 2019, CC BY 4.0)
- Requires an Earth Engine account and a Cloud project (set `PROJECT_ID` to your own)

**Method:** Layers are first averaged from their native resolution (250 m soil and salinity, 30 m terrain) to the 2 km grid, then sampled at each grid point on Earth Engine. Results are exported as small CSV tiles and merged here.
- **Soil (9 properties x 2 depths = 18 features):** pH, organic carbon, clay, sand, silt, CEC, nitrogen, bulk density and coarse fragments, each as a **topsoil** (0-30 cm) and **subsoil** (30-100 cm) thickness-weighted average, converted to conventional units
- **Terrain (5 features):** elevation, slope, northness, eastness and elevation roughness
- **Salinity (5 features):** class value in the 2016 map (mean and share of pixels with class >= 1), mean and maximum across seven maps (1986-2016), and change 1986-2016
- Masked pixels are filled with -9999 during export and converted to NaN in the merge, so no grid cell is dropped

**Outputs** (stored in Google Drive, not in this repo):
- `soil_terrain.parquet`: 23 float32 features per `grid_id`
- `salinity.parquet`: 5 float32 features per `grid_id`

**Notes:**
- **Salinity is a modelled product, not a measurement.** The global maps come from a random forest trained on lab ECe measurements (WoSIS) plus satellite and soil covariates, with a reported validation accuracy of 67-70%. In the Sahel, most cells show no salinity signal, so these features act as a sparse hotspot flag.
- SoilGrids values are also modelled predictions and are less reliable in data-poor areas.
- Salinity class definitions are taken from the original paper; see Ivushkin et al. (2019), doi 10.1016/j.rse.2019.111260.
- Share of NaN values per feature is reported at the end of each merge step.
- Salinity is also approximated from Sentinel-2 spectral indices in notebook 04, as an independent proxy.

In [ ]:
# @title Cell 1: setup
import ee, os, glob
import pandas as pd
from google.colab import drive

PROJECT_ID = 'your-project-id'
GRID_SCALE = 2000
OUT_FOLDER = 'crop_project_soil'     # new Drive folder for this notebook's exports

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'
grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
print(grid.shape)

In [ ]:
# @title Cell 2: check band names before spending quota
props = ['phh2o', 'soc', 'clay', 'sand', 'silt', 'cec', 'nitrogen', 'bdod', 'cfvo']
for p in props:
    img = ee.Image(f'projects/soilgrids-isric/{p}_mean')
    print(p, img.bandNames().getInfo())

print('SRTM:', ee.Image('USGS/SRTMGL1_003').bandNames().getInfo())

In [ ]:
# @title Cell 3: build the soil and terrain image
proj = ee.Projection('EPSG:4326').atScale(GRID_SCALE)

# SoilGrids stored-integer scaling factors (divide stored value by this)
scale = {'phh2o': 10, 'soc': 10, 'clay': 10, 'sand': 10, 'silt': 10,
         'cec': 10, 'nitrogen': 100, 'bdod': 100, 'cfvo': 10}

top_depths = ['0-5cm', '5-15cm', '15-30cm']      # thickness 5, 10, 15 cm
top_w      = [5, 10, 15]
sub_depths = ['30-60cm', '60-100cm']             # thickness 30, 40 cm
sub_w      = [30, 40]

def weighted(p, depths, weights):
    img = ee.Image(f'projects/soilgrids-isric/{p}_mean')
    total = sum(weights)
    acc = None
    for d, w in zip(depths, weights):
        b = img.select(f'{p}_{d}_mean').multiply(w)
        acc = b if acc is None else acc.add(b)
    return acc.divide(total).divide(scale[p])

soil_bands = []
for p in scale:
    soil_bands.append(weighted(p, top_depths, top_w).rename(f'{p}_top'))
    soil_bands.append(weighted(p, sub_depths, sub_w).rename(f'{p}_sub'))
soil = ee.Image.cat(soil_bands).float()

# Average 250 m soil to the 2 km grid
soil_2km = (soil.reduceResolution(ee.Reducer.mean(), maxPixels=1024)
                .reproject(proj))

# ---------- Terrain (SRTM 30 m) ----------
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope = ee.Terrain.slope(dem).rename('slope_deg')
asp = ee.Terrain.aspect(dem).multiply(3.14159265 / 180)
north = asp.cos().rename('northness')
east  = asp.sin().rename('eastness')

terrain_mean = (dem.addBands([slope, north, east])
                .reduceResolution(ee.Reducer.mean(), maxPixels=8192)
                .reproject(proj)
                .rename(['elevation', 'slope_deg', 'northness', 'eastness']))
elev_std = (dem.reduceResolution(ee.Reducer.stdDev(), maxPixels=8192)
               .reproject(proj).rename('elevation_std'))

# -9999 marks masked pixels so no grid cell is silently dropped; replaced by NaN in the merge
soil_terrain = ee.Image.cat([soil_2km, terrain_mean, elev_std]).float().unmask(-9999)
print(len(soil_terrain.bandNames().getInfo()), 'bands')
print(soil_terrain.bandNames().getInfo())

In [ ]:
# @title Cell 4: test on a few points
def to_fc(d):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'grid_id': r.grid_id})
        for r in d.itertuples()])

test_pts = grid.sample(6, random_state=0)
res = soil_terrain.sampleRegions(collection=to_fc(test_pts), properties=['grid_id'],
                                 scale=GRID_SCALE, projection=proj, tileScale=4).getInfo()
out = pd.DataFrame([f['properties'] for f in res['features']]).set_index('grid_id')
out.insert(0, 'adm0', test_pts.set_index('grid_id').adm0)
pd.set_option('display.width', 200)
print(out.T.round(2))

In [ ]:
# @title Cell 5: start exports
BLOCK, MAXPTS = 2, 8000

g = grid[['grid_id', 'lon', 'lat']].copy()
g['bx'] = (g.lon // BLOCK * BLOCK).astype(int)
g['by'] = (g.lat // BLOCK * BLOCK).astype(int)

jobs = []
for (bx, by), d in g.groupby(['bx', 'by']):
    for i in range(0, len(d), MAXPTS):
        jobs.append((f'soil_x{bx}_y{by}_p{i // MAXPTS}', d.iloc[i:i + MAXPTS]))
print(len(jobs), 'export jobs for', len(g), 'points')

already = {t.config.get('description') for t in ee.batch.Task.list()[:2000]}
started = 0
for name, d in jobs:
    if name in already:
        continue
    ee.batch.Export.table.toDrive(
        collection=soil_terrain.sampleRegions(collection=to_fc(d), properties=['grid_id'],
                                              scale=GRID_SCALE, projection=proj, tileScale=4),
        description=name, folder=OUT_FOLDER, fileFormat='CSV').start()
    started += 1
print('Started', started, 'tasks')

In [ ]:
# @title Cell 6: monitor
tasks = [t for t in ee.batch.Task.list()[:2000]
         if t.config.get('description', '').startswith('soil_')]
counts = {}
for t in tasks:
    s = t.status()
    counts[s['state']] = counts.get(s['state'], 0) + 1
    if s['state'] == 'FAILED':
        print('FAILED:', s['description'], '->', s.get('error_message'))
print(counts)

In [ ]:
# @title Cell 7: merge into "soil_terrain.parquet"
import numpy as np

files = glob.glob(f'{BASE}/{OUT_FOLDER}/*.csv')
parts = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
st = pd.concat(parts, ignore_index=True)
st = st.drop(columns=['system:index', '.geo'], errors='ignore')
st = st.drop_duplicates('grid_id').reset_index(drop=True)
print(len(files), 'files ->', st.shape)

feat = [c for c in st.columns if c != 'grid_id']
st[feat] = st[feat].replace(-9999, np.nan).astype('float32')

missing = grid[~grid.grid_id.isin(st.grid_id)]
print(f'{len(missing)} of {len(grid)} grid cells have no row')
print('NaN share per column:'); print((st[feat].isna().mean() * 100).round(1))

st.to_parquet(f'{BASE}/crop_project/soil_terrain.parquet', index=False)
print(st[feat].describe(percentiles=[.05, .5, .95]).T.round(2))

In [ ]:
# @title Cell 8: Extracts Soil salinity

import ee
import pandas as pd
from google.colab import drive

PROJECT_ID = 'crop-suitability-511809'
GRID_SCALE = 2000

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'
grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')

proj = ee.Projection('EPSG:4326').atScale(GRID_SCALE)

def to_fc(d):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'grid_id': r.grid_id})
        for r in d.itertuples()])

# ---------- Inspect the salinity layer ----------
sal = ee.ImageCollection('projects/sat-io/open-datasets/global_soil_salinity')
first = sal.first()

print('Images in collection:', sal.size().getInfo())
print('Image ids:', sal.aggregate_array('system:index').getInfo())
print('Bands of first image:', first.bandNames().getInfo())
print('Properties of first image:', first.propertyNames().getInfo())
print('Nominal scale (m):', first.select(0).projection().nominalScale().getInfo())

# Sample the all-years mean at 300 random grid points to see what the values look like
pts = grid.sample(300, random_state=1)
res = (sal.select([0]).mean()
       .sampleRegions(collection=to_fc(pts), properties=['grid_id'],
                      scale=GRID_SCALE, projection=proj, tileScale=4).getInfo())
vals = pd.DataFrame([f['properties'] for f in res['features']])
print('\nPoints with a value:', len(vals), 'of 300')
print(vals.drop(columns='grid_id').describe().round(2))
print(vals.drop(columns='grid_id').iloc[:, 0].round(0).value_counts().sort_index())

In [ ]:
# @title Cell 9: build the salinity features

OUT_FOLDER = 'crop_project_soil'      # same Drive folder as the soil exports

sal = ee.ImageCollection('projects/sat-io/open-datasets/global_soil_salinity')
P250 = sal.first().select('b1').projection()
YEARS = [1986, 1992, 2000, 2002, 2005, 2009, 2016]

def year_img(y):
    # mosaic the tiles of one year; keep the native 250 m projection
    return (sal.filter(ee.Filter.stringStartsWith('system:index', f'salMap{y}'))
            .mosaic().select('b1').setDefaultProjection(P250))

yimgs = [year_img(y) for y in YEARS]
all_years = ee.ImageCollection(yimgs)
mean_all = all_years.mean().setDefaultProjection(P250)
max_all  = all_years.max().setDefaultProjection(P250)
s2016, s1986 = yimgs[-1], yimgs[0]
any2016 = s2016.gte(1).setDefaultProjection(P250)

def to_2km(img, reducer, name):
    return (img.reduceResolution(reducer, maxPixels=1024)
               .reproject(proj).rename(name))

sal_mean_2016  = to_2km(s2016, ee.Reducer.mean(), 'sal_mean_2016')
sal_share_2016 = to_2km(any2016, ee.Reducer.mean(), 'sal_share_2016')
sal_mean_all   = to_2km(mean_all, ee.Reducer.mean(), 'sal_mean_allyears')
sal_max_all    = to_2km(max_all, ee.Reducer.max(), 'sal_max_allyears')
sal_change     = sal_mean_2016.subtract(
                     to_2km(s1986, ee.Reducer.mean(), 'tmp')).rename('sal_change_1986_2016')

# -9999 marks masked pixels; converted to NaN in the merge
sal_img = (ee.Image.cat([sal_mean_2016, sal_share_2016, sal_mean_all, sal_max_all, sal_change])
           .float().unmask(-9999))
print(sal_img.bandNames().getInfo())

In [ ]:
# @title Cell 10: test on a sample
import numpy as np

pts = grid.sample(300, random_state=2)
res = sal_img.sampleRegions(collection=to_fc(pts), properties=['grid_id'],
                            scale=GRID_SCALE, projection=proj, tileScale=4).getInfo()
t = pd.DataFrame([f['properties'] for f in res['features']]).set_index('grid_id')
t = t.replace(-9999, np.nan)
print('Rows returned:', len(t), 'of 300')
print(t.describe().T.round(3))
print('\nShare of cells with any signal:')
print((t[['sal_share_2016', 'sal_max_allyears']] > 0).mean().round(3))

In [ ]:
# @title Cell 11: start exports
BLOCK, MAXPTS = 2, 8000

g = grid[['grid_id', 'lon', 'lat']].copy()
g['bx'] = (g.lon // BLOCK * BLOCK).astype(int)
g['by'] = (g.lat // BLOCK * BLOCK).astype(int)

jobs = []
for (bx, by), d in g.groupby(['bx', 'by']):
    for i in range(0, len(d), MAXPTS):
        jobs.append((f'sal_x{bx}_y{by}_p{i // MAXPTS}', d.iloc[i:i + MAXPTS]))
print(len(jobs), 'export jobs for', len(g), 'points')

already = {t.config.get('description') for t in ee.batch.Task.list()[:2000]}
started = 0
for name, d in jobs:
    if name in already:
        continue
    ee.batch.Export.table.toDrive(
        collection=sal_img.sampleRegions(collection=to_fc(d), properties=['grid_id'],
                                         scale=GRID_SCALE, projection=proj, tileScale=4),
        description=name, folder=OUT_FOLDER, fileFormat='CSV').start()
    started += 1
print('Started', started, 'tasks')

In [ ]:
# @title Cell 13: merge into salinity.parquet
import glob, os

files = glob.glob(f'{BASE}/{OUT_FOLDER}/sal_*.csv')
parts = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
sl = pd.concat(parts, ignore_index=True)
sl = (sl.drop(columns=['system:index', '.geo'], errors='ignore')
        .drop_duplicates('grid_id').reset_index(drop=True))

feat = [c for c in sl.columns if c != 'grid_id']
sl[feat] = sl[feat].replace(-9999, np.nan).astype('float32')
print(len(files), 'files ->', sl.shape)

missing = grid[~grid.grid_id.isin(sl.grid_id)]
print(f'{len(missing)} of {len(grid)} grid cells have no row')
print('NaN share per column (%):'); print((sl[feat].isna().mean() * 100).round(1))

sl.to_parquet(f'{BASE}/crop_project/salinity.parquet', index=False)

# Where does the map show salinity? (share of cells with any signal, by country)
chk = grid[['grid_id', 'adm0']].merge(sl, on='grid_id')
print((chk.assign(any_signal=chk.sal_max_allyears > 0)
          .groupby('adm0').any_signal.mean() * 100).round(1).sort_values(ascending=False))